# Market Basket Analysis: Apriori & ECLAT on Bakery Transactions

This notebook mines **association rules** from real point-of-sale transactions
of a French bakery, using two classic algorithms:

- **Apriori** (via `mlxtend`) — the standard candidate-generation approach
- **ECLAT** — a vertical, set-intersection approach implemented from scratch

The goal is to discover which products tend to be bought together, so the
bakery could use the results for product placement, bundling, or recommendations.

**Dataset:** `Bakery.csv` — 20,507 line items across 9,465 transactions,
collected between Oct 2016 and Apr 2017.


In [ ]:
# Imports
import itertools
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_colwidth", None)


## 1. Loading and Exploring the Data

In [ ]:
bakery = pd.read_csv("Bakery.csv")
print("Shape:", bakery.shape)
print("Unique transactions:", bakery["TransactionNo"].nunique())
print("Unique items:", bakery["Items"].nunique())
bakery.head()


In [ ]:
# Parse the datetime column and pull out useful calendar features
bakery["DateTime"] = pd.to_datetime(bakery["DateTime"])
bakery["Day"] = bakery["DateTime"].dt.day_name()
bakery["Month"] = bakery["DateTime"].dt.month_name()
bakery.head()


### 1.1 Best-selling items

In [ ]:
item_frequency = bakery["Items"].value_counts().head(15)

item_frequency.sort_values().plot(kind="barh", color="#3fa796")
plt.title("15 Most Frequent Items")
plt.xlabel("Number of transactions")
plt.ylabel("")
plt.tight_layout()
plt.show()


Coffee is the clear best-seller, followed by bread and tea — a typical morning-bakery profile.

### 1.2 Sales by day part and day of week

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

daypart_counts = bakery.groupby("Daypart")["Items"].count().sort_values(ascending=False)
axes[0].pie(daypart_counts, labels=daypart_counts.index, autopct="%1.0f%%",
            colors=plt.cm.BuGn(np.linspace(0.4, 0.8, len(daypart_counts))))
axes[0].set_title("Sales by Day Part")

day_counts = bakery.groupby("Day")["Items"].count().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
)
axes[1].bar(day_counts.index, day_counts.values, color="#3fa796")
axes[1].set_title("Sales by Day of Week")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


Most sales happen in the afternoon, and weekends (Saturday/Sunday) clearly outsell weekdays.

## 2. Preparing Transactions

Both Apriori and ECLAT need the data as a **list of transactions**, where each
transaction is the set of distinct items bought together (duplicates within a
transaction — e.g. two coffees — are dropped, since association mining cares
about co-occurrence, not quantity).


In [ ]:
transactions = (
    bakery.groupby("TransactionNo")["Items"]
    .apply(lambda items: sorted(set(items)))
    .tolist()
)

n_transactions = len(transactions)
print(f"{n_transactions} transactions prepared")
transactions[:5]


In [ ]:
# One-hot encode for mlxtend's Apriori implementation
te = TransactionEncoder()
encoded = te.fit(transactions).transform(transactions)
onehot = pd.DataFrame(encoded, columns=te.columns_)
onehot.head()


## 3. Apriori

We mine frequent itemsets with a minimum support of 2% (i.e. bought together
in at least 2% of all transactions), then generate rules and rank them by
**lift** — how much more likely the consequent is, given the antecedent,
compared to chance.


In [ ]:
frequent_itemsets = apriori(onehot, min_support=0.02, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False).reset_index(drop=True)
frequent_itemsets.head(10)


In [ ]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)
rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)


### 3.1 Refining the rules

Coffee is bought so often that almost anything shows reasonable confidence
paired with it, even without a real association. We drop rules where Coffee
is the consequent so the more meaningful (higher-lift) pairings stand out.


In [ ]:
refined_rules = rules[rules["consequents"] != "Coffee"].sort_values("lift", ascending=False)
refined_rules = refined_rules[["antecedents", "consequents", "support", "confidence", "lift"]].reset_index(drop=True)
refined_rules.head(10)


### 3.2 Visualizing the rule network

In [ ]:
top_rules = refined_rules.head(15)

G = nx.DiGraph()
for _, row in top_rules.iterrows():
    G.add_edge(row["antecedents"], row["consequents"], weight=row["lift"])

pos = nx.spring_layout(G, seed=42, k=0.9)
plt.figure(figsize=(10, 8))
nx.draw_networkx_nodes(G, pos, node_color="#3fa796", node_size=1800, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=8)
nx.draw_networkx_edges(G, pos, arrowstyle="-|>", arrowsize=15, edge_color="gray")
plt.title("Top 15 Association Rules by Lift")
plt.axis("off")
plt.tight_layout()
plt.show()


## 4. ECLAT

ECLAT takes a different route: instead of scanning the transaction list
repeatedly (as a naive/hand-rolled implementation would), it converts the data
into a **vertical format** — for every item, the set of transaction IDs
("tidset") it appears in — and finds frequent itemsets by **intersecting**
tidsets. This is both faster and much simpler than filtering strings with
`eval()`.


In [ ]:
# Vertical (item -> tidset) representation
item_tidsets = defaultdict(set)
for tid, items in enumerate(transactions):
    for item in items:
        item_tidsets[item].add(tid)

min_support_count = 0.02 * n_transactions
frequent_items = {item: tids for item, tids in item_tidsets.items() if len(tids) >= min_support_count}

print(f"{len(frequent_items)} items meet the 2% support threshold")


In [ ]:
def eclat_combinations(frequent_items, k, n_transactions, min_support_count):
    """Score every k-item combination of the frequent items by ECLAT
    (tidset intersection) and return a DataFrame sorted by support."""
    rows = []
    for combo in itertools.combinations(sorted(frequent_items), k):
        tids = frequent_items[combo[0]]
        for item in combo[1:]:
            tids = tids & frequent_items[item]
            if len(tids) < min_support_count:
                break
        if len(tids) >= min_support_count:
            rows.append({"Itemset": ", ".join(combo), "Support": len(tids) / n_transactions})
    result = pd.DataFrame(rows, columns=["Itemset", "Support"])
    return result.sort_values("Support", ascending=False).reset_index(drop=True)


eclat_pairs = eclat_combinations(frequent_items, 2, n_transactions, min_support_count)
eclat_pairs.head(10)


Note: 3-item combinations are naturally rarer than pairs, so a 2% threshold
leaves nothing. We relax it to 0.5% support just for trios, computed against
the same vertical tidsets (no need to rebuild anything).


In [ ]:
trio_min_support_count = 0.005 * n_transactions
eclat_trios = eclat_combinations(item_tidsets, 3, n_transactions, trio_min_support_count)
eclat_trios.head(10)


## 5. Apriori vs. ECLAT: Do They Agree?

Apriori and ECLAT are different algorithms for the same underlying question,
so their top **frequent pairs** (by support) should match closely — the main
difference is that Apriori additionally gives us confidence/lift, while ECLAT
just gives support.


In [ ]:
apriori_pairs = frequent_itemsets[frequent_itemsets["itemsets"].apply(len) == 2].copy()
apriori_pairs["Itemset"] = apriori_pairs["itemsets"].apply(lambda s: ", ".join(sorted(s)))
apriori_pairs = apriori_pairs[["Itemset", "support"]].sort_values("support", ascending=False).reset_index(drop=True)

comparison = apriori_pairs.head(10).merge(
    eclat_pairs.head(10), on="Itemset", how="outer", suffixes=("_apriori", "_eclat")
)
comparison


As expected, both algorithms surface the same top pairs — **(Bread, Coffee)**
being the strongest — since they're computing the same statistic (support)
via different mechanics.


## 6. Summary

- **Coffee, Bread, and Tea** dominate individual sales, and most of that is
  simply because they're purchased constantly — not because of a strong
  association with other items. Filtering out Coffee as a consequent was
  necessary to see meaningful pairings.
- The most actionable pairing is **(Bread, Coffee)**, which shows up as the
  top itemset in both Apriori and ECLAT.
- Apriori and ECLAT agree on frequent itemsets (as they must — they estimate
  the same support values), but Apriori is more convenient here since
  `mlxtend` also derives confidence and lift directly.
- Sales peak in the **afternoon** and on **weekends**, which is useful context
  for staffing and promotions alongside the product-pairing insights above.

### Possible extensions
- Segment the analysis by `Daypart` or `Day` to find time-specific pairings
  (e.g. what pairs well with breakfast vs. afternoon treats).
- Compare rules generated with different `min_support` / `min_threshold`
  values to see how sensitive the recommendations are.
- Turn the top rules into a simple "customers who bought X also bought Y"
  recommender.
